In [ ]:
# Datasets laden
from datasets import load_dataset
import torch
# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("liweijiang/infinite-chats-eval", split="train")

torch.cuda.is_available()

In [ ]:
from transformers import pipeline
import torch
from transformers import GenerationConfig, AutoTokenizer

print(torch.version.cuda)
device = 0 if torch.cuda.is_available() else -1
print(f"using device = {'cuda' if device == 0 else 'cpu'}")
with_system = False

models = ["mistralai/Ministral-8B-Instruct-2410",
          #"google/gemma-2-9b-it", 
          "Qwen/Qwen3-8B", 
          "mistralai/Mistral-7B-Instruct-v0.1", 
          "CohereForAI/aya-expanse-8b",
          #"allenai/OLMo-2-1124-7B-Instruct", # noch nicht generiert
          "allenai/Llama-3.1-Tulu-3-8B",
          "meta-llama/Llama-3.1-8B-Instruct",
          "meta-llama/Llama-3.2-3B-Instruct",  
          "Qwen/Qwen2.5-14B-Instruct",
          "Qwen/Qwen1.5-14B-Chat",
          "microsoft/phi-4",
          #"google/gemma-1.1-7b-it",
          "mistralai/Mistral-Nemo-Instruct-2407",
         ]
for model_id in models:
    tokenizer = AutoTokenizer.from_pretrained(model_id) 
    
    def format_query(query):
        if with_system == True:
            messages_with_system = [
                {"role": "system", "content": "Answer the questions with a high degree of ingenuity and creativity, and don’t hesitate to draw on unconventional ideas."},
                {"role": "user", "content": query}
            ]
            return tokenizer.apply_chat_template(
                messages_with_system,
                tokenize=False,
                add_generation_prompt=True  
            )
        else:
            messages_user_only = [
                {"role": "user", "content": query}
            ]
            return tokenizer.apply_chat_template(
                    messages_user_only,
                    tokenize=False,
                    add_generation_prompt=True
            )
    
    def config_Generation():
        
        gen_cfg = GenerationConfig(
            max_new_tokens=2048,  # maximale Antwortlänge
            eos_token_id=tokenizer.eos_token_id, # damit End-of-sequence Generierung beendet!!
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            num_return_sequences=10,
            temperature=1.0,
            top_p=0.9
        )
        return gen_cfg

    pipe = pipeline(
        task="text-generation",
        model=model_id,
        tokenizer=tokenizer,
        device=device,
    )
    pipe.generation_config = config_Generation()
    
    model_name = model_id.split("/")[1]
    print(model_name)
    out_file = [f"answers_{model_name}_Teil1.txt",
                f"answers_{model_name}_Teil2.txt", 
                f"answers_{model_name}_Teil3.txt", 
                f"answers_{model_name}_Teil4.txt", 
                f"answers_{model_name}_Teil5.txt"]
    batch_size = 2
    for out in out_file:
        print(out)
        with open(out, "a", encoding="utf-8") as f:
            for example in dataset:
                query = example["query"]
                prompt = format_query(query)
        
                outputs = pipe(
                    prompt,
                    batch_size=batch_size,
                    return_full_text=False,   # nur Antwort, nicht Prompt
                    truncation=True,
                )
                for out in outputs:
                    text = out["generated_text"]
                    f.write("\nnewOutput:\n" + text)
                print("output")
